# Evaluating Observed Flood Extent from the GFM against Forecast Flood Extent from the EFAS or GloFAS Rapid Flood Mapping layer

With this notebook, we show how to use the new GFM STAC service to
derive the maximum flood extent for a flood event, extract the corresponding
forecast product from the EFAS or GloFAS Rapid Flood Mapping (RFM) layer and perform
a simple evaluation.

This example notebook is structured as follows:

0. (Import and install the necessary python packages)
1. Download data from the GFM STAC service (ensemble flood extent, reference water, exclusion mask)
2. Download the EFAS or GloFAS Rapid Flood Mapping data
3. Evaluate the RFM layer against the GFM data - exclude areas either within reference water mask or exclusion masks

We present an example analysis of the flooding that occurred in southern
Mozambique between the 1st - 31st January 2026

Please note you must have an EFAS account to access EFAS data.
GloFAS data is publicly accessible and does not require an account.

## 0. Install and import necessary python modules

In [ ]:
!pip install pyproj xarray shapely pystac_client odc-stac matplotlib rioxarray geopandas ipykernel

In [1]:
# Some necessary imports

from pystac_client import Client
from datetime import datetime
from odc import stac as odc_stac
from pathlib import Path
import pyproj
import rioxarray # noqa
import xarray as xr
from shapely.geometry import box
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds

import numpy as np
import requests
from zipfile import ZipFile

## 1. Extract GFM Data using the GFM STAC service

Firstly define the GFM product you wish to download.

Then define a time range of interest and the coordinates of the area of
interest (AOI). The coordinates of the AOI can be retrieved either via the GFM Portal (portal.gfm.eodc.eu) or a tool like bboxfinder.com.

These parameters are passed to the GFM STAC service to find the available data.

In [3]:
# Define the output path, where you want to save temporary data and final results
output_path = Path("./gfm_rfm_evaluation")
output_path.mkdir(exist_ok=True)

# Define bounding box (West, South, East, North) - Southern Mozambique
aoi = box(32.048307386, -25.272982457, 34.109853235, -23.221383919)

# Define time range
time_range = (datetime(2026, 1, 7), datetime(2026, 1, 28))

# EODC STAC API URL
eodc_catalog = Client.open("https://stac.eodc.eu/api/v1")

# Define search query using pystac_client
search = eodc_catalog.search(
    max_items=1000,
    collections=["GFM"],
    bbox=aoi.bounds,
    datetime=time_range
)

# Get all items
items = search.item_collection()
if not items:
    raise RuntimeError("No items found — check your AOI, time range, or collection name.")

print(f"Found {len(items)} items matching filter criteria.")

# Retrieve the coordinate reference system (CRS) from one of the found items
crs_equi7 = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"])

Found 92 items matching filter criteria.


### Lazy-load the returned data
Now the returned data are lazily loaded into an xarray dataset in the Equi7Grid projection.

We define the asset names we want to use in this notebook. In our case, that is 'ensemble_flood_extent', 'reference_water_mask' and 'exclusion_mask'.

In [ ]:
# Define the asset names you want to use 
asset_names = ["ensemble_flood_extent", "reference_water_mask", "exclusion_mask"]

# Retrieve the resolution of the data
resolution = items[0].properties.get("gsd", 20)  # fallback to 20m if gsd missing

# By setting chunks, the odc library "lazy loads" the data. -1 for time means
# that all time steps are included in one chunk. Reduce the chunk size for x
# and/or y if your kernel crashes due to out of memory issues
raw_data = odc_stac.load(
    items, 
    bbox=aoi.bounds,   # Define the bounding box for the area of interest
    crs=crs_equi7,   # Set the coordinate reference system
    bands=asset_names,   # Specify the bands to load
    resolution=resolution,   # Set the resolution of the data
    dtype='uint8',   # Define the data type
    groupby="solar_day",
    chunks={"x": 2048, "y": 2048, "time": -1}, 
)

## 2. Save the downloaded data as a Zarr store for later use (optional)
Once the above cell is loaded, we advise to save the data on disk for later use.
With that, you do not need to download the same data everytime you start this notebook.

### Save the downloaded data as a Zarr store

In [ ]:
# Save the "raw" data as ZARR data store for later use
zarr_store = Path(output_path) / "./gfm_data.zarr"
# zarr_store.unlink(missing_ok=True)  # Remove existing ZARR store if it exists
# raw_data.to_zarr(zarr_store)

### Load the data from the Zarr store once saved

In [18]:
# Read data from ZARR data store once saved
raw_data = xr.open_zarr(zarr_store)

# ZARR is currently not writing CRSs into data stores. Add it again after
# opening the data store
raw_data.rio.write_crs(crs_equi7, inplace=True)

<xarray.Dataset> Size: 6GB
Dimensions:                (time: 15, y: 11543, x: 11619)
Coordinates:
  * time                   (time) datetime64[ns] 120B 2026-01-07T03:01:42 ......
  * y                      (y) float64 92kB 2.478e+06 2.478e+06 ... 2.247e+06
  * x                      (x) float64 93kB 6.747e+06 6.747e+06 ... 6.98e+06
    spatial_ref            int64 8B 0
Data variables:
    ensemble_flood_extent  (time, y, x) uint8 2GB dask.array<chunksize=(15, 2048, 2048), meta=np.ndarray>
    exclusion_mask         (time, y, x) uint8 2GB dask.array<chunksize=(15, 2048, 2048), meta=np.ndarray>
    reference_water_mask   (time, y, x) uint8 2GB dask.array<chunksize=(15, 2048, 2048), meta=np.ndarray>

## 3. Mask and save data as GeoTiff

We mask the data to exclude invalid values such as nodata (=255) and non-flood (=0).

After masking, we save a GeoTiff file for each GFM layer for easy visualization in e.g. QGIS.

In [19]:
# Only include data which is not nodata and not 0
nodata = 255
valid_data = raw_data.where((raw_data != nodata) & (raw_data != 0))

# Create the sum over the time dimension
# This is done for all specified assets
data = valid_data.sum(dim="time").astype("uint8")

In [20]:
# Extract maximum flood extent
data["ensemble_flood_extent"] = data["ensemble_flood_extent"].where((data["ensemble_flood_extent"] < 1) | (data["ensemble_flood_extent"] > nodata), 1)

# Reference water mask
data["reference_water_mask"] = data["reference_water_mask"].where((data["reference_water_mask"] < 1) | (data["reference_water_mask"]  > nodata), 1)

# Exclusion mask
data["exclusion_mask"] = data["exclusion_mask"].where((data["exclusion_mask"] < 1) | (data["exclusion_mask"] > nodata), 1)

# Save as Raster in Equi7Grid
data["ensemble_flood_extent"].rio.to_raster("ensemble_flood_extent_equi7.tif", compress="LZW", tiled=True, blockxsize=512, blockysize=512)
data["reference_water_mask"].rio.to_raster("reference_water_mask_equi7.tif", compress="LZW", tiled=True, blockxsize=512, blockysize=512)
data["exclusion_mask"].rio.to_raster("exclusion_mask_equi7.tif", compress="LZW", tiled=True, blockxsize=512, blockysize=512)

## 4. Download Forecasted Flood Extent from the EFAS or GloFAS Rapid Flood Mapping layer

Specify the data `source` (`"efas"` or `"glofas"`) and the date of the forecast.

For EFAS, you will also need a web token, which you can generate by logging in to
https://european-flood.emergency.copernicus.eu/en/efas-web-services and copying
the web token shown in the table. Note: the token needs to be regenerated after every use.

GloFAS data is publicly accessible and does not require a token.

In [22]:
source = "glofas"  # Set to "efas" or "glofas"
fcast_date = '2026010700'  # date of the forecast as string in YYYYMMDDHH format

# EFAS requires a token; GloFAS does not
# only needed when source == "efas", token must be regenerated after every use
efas_token = 'your_efas_token'                                

date_str = f'{fcast_date[0:4]}-{fcast_date[4:6]}-{fcast_date[6:8]}T{fcast_date[8:10]}:00Z'
base_url = f'https://european-flood.emergency.copernicus.eu/api/fms/download/{source}/RapidFloodMapping/{date_str}'
url_rfm = f'{base_url}?token={efas_token}' if source == "efas" else base_url

#### Download the forecast flood extent file, load it in then clip to AOI

This will save a zip file into the same folder where you have saved
this notebook. An ESRI shapefile containing the forecast flood 
extent will be extracted from this zip file into the same folder.

In [24]:
response = requests.get(url_rfm)
response.raise_for_status()

out_zip = output_path / f'RFM_{fcast_date}.zip'
with open(out_zip, 'wb') as file:
  file.write(response.content)

with ZipFile(out_zip, 'r') as zip_ref:
  zip_ref.extractall(output_path)

#### Load the Forecasted Flood Extent file

The forecast flood extent shapefile is loaded as a GeoPandas
GeoDataFrame. It is then clipped to the extent of the aoi 
which you defined previously

In [27]:
rfm_file = output_path / f'FloodMaskMerged{fcast_date}.shp'
rfm_data = gpd.read_file(rfm_file)

# Clip the Rapid Flood Mapping to your drawn area
rfm_intersect = gpd.clip(rfm_data, gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[aoi]))

rfm_file = output_path / f'FloodMask{fcast_date}.shp'
rfm_data = gpd.read_file(rfm_file)

# Clip the Rapid Flood Mapping to your drawn area
rfm_intersect = gpd.clip(rfm_data, gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[aoi]))

if rfm_intersect.empty:
    raise RuntimeError("No RFM data found within the AOI — check your bounding box or forecast date.")

In [28]:
# Get the bounding coordinates of the GFM data
dx = data.x.values[1]-data.x.values[0]
dy = data.y.values[1]-data.y.values[0]

xmin = data.x.values.min() - (dx/2)
xmax = data.x.values.max() + (dx/2)

ymin = data.y.values.min() - abs(dy/2)
ymax = data.y.values.max() + abs(dy/2)

In [29]:
gfm_flood = data["ensemble_flood_extent"].values

transform = from_bounds(xmin, ymin, xmax, ymax, gfm_flood.shape[1], gfm_flood.shape[0])

rfm_intersect_equi7 = rfm_intersect.to_crs(crs_equi7)

# Get geometry and corresponding value from the GeoDataFrame
shapes = ((geom, 1) for geom in rfm_intersect_equi7.geometry)

# Rasterize the geometries into the numpy array
rfm_raster = rasterize(
    shapes,
    out_shape=gfm_flood.shape,
    transform=transform,
    fill=0, 
    dtype=np.uint8
)

#### Compare the Observed and Forecast Flood Extent Grids

Use a contingency table approach to classify each grid cell as one of the following:
1. Hit - where flooding is present in both the GFM & RFM forecast
2. False alarm - flooding only present in the RFM forecast
3. Miss - flooding only present in the GFM observations
4. Correct negative - no flooding present in either GFM or RFM data

We exclude areas which are either a) in the reference water mask, or b) in the exclusion mask

From this classification, we compute the following performance scores:
1. CSI (Critical Success Index), range 0-1 where 1 = perfect agreement between forecast and observations
2. False Alarm Ratio, range 0-1 where 0 is optimal, shows the fraction of grid cells which were forecasted as being flooded which were incorrect
3. Hit Rate, range 0-1 where 1 is optimal, shows the fraction of observed flooded grid cells which were correctly forecasted

In [ ]:
# compare GFM and RFM data

gfm_rwm = data["reference_water_mask"].values
gfm_em = data["exclusion_mask"].values

# compute the total number of hits, false alarms, misses and correct negatives
tp = np.where((gfm_flood == 1) & (rfm_raster == 1) & (gfm_rwm == 0) & (gfm_em == 0))[0].shape[0] # hits
fp = np.where((gfm_flood == 0) & (rfm_raster == 1) & (gfm_rwm == 0) & (gfm_em == 0))[0].shape[0] # false alarms
fn = np.where((gfm_flood == 1) & (rfm_raster == 0) & (gfm_rwm == 0) & (gfm_em == 0))[0].shape[0] # misses
tn = np.where((gfm_flood == 0) & (rfm_raster == 0) & (gfm_rwm == 0) & (gfm_em == 0))[0].shape[0] # correct negatives

csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float('nan')  # critical success index (CSI)
far = fp / (tp + fp) if (tp + fp) > 0 else float('nan')            # false alarm ratio (FAR)
hr  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')            # hit rate (HR)

print(f'CSI: {"%.2f" % csi}')
print(f'False Alarm Ratio: {"%.2f" % far}')
print(f'Hit Rate: {"%.2f" % hr}')

# Create a grid for the evaluation results to show on a map
# create a new grid which shows whether each grid cell is a hit, false alarm or miss
eval_grid = np.zeros(gfm_flood.shape, dtype=np.uint8)
eval_grid[(gfm_flood == 1) & (rfm_raster == 1) & (gfm_rwm == 0) & (gfm_em == 0)] = 1  # Hits
eval_grid[(gfm_flood == 1) & (rfm_raster == 0) & (gfm_rwm == 0) & (gfm_em == 0)] = 2  # Misses
eval_grid[(gfm_flood == 0) & (rfm_raster == 1) & (gfm_rwm == 0) & (gfm_em == 0)] = 3  # False alarms

In [ ]:
# Save RFM data as raster
with rasterio.open(output_path / 'rfm_data_equi7.tif', 'w', driver='GTiff', height=rfm_raster.shape[0],
                   width=rfm_raster.shape[1], count=1, dtype=rfm_raster.dtype,
                   crs=crs_equi7, transform=transform, compress="LZW") as dst:
    dst.write(rfm_raster, 1)

# Save RFM GFM evaluation results as raster
with rasterio.open(output_path / 'rfm_gfm_eval_equi7.tif', 'w', driver='GTiff', height=eval_grid.shape[0],
                   width=eval_grid.shape[1], count=1, dtype=eval_grid.dtype,
                   crs=crs_equi7, transform=transform, compress="LZW") as dst:
    dst.write(eval_grid, 1)

## Conclusions

This notebook shows how you can download specific GFM data 
using the GFM STAC service and the EFAS or GloFAS Rapid Flood Mapping forecast 
layer and compare the data, with the idea of evaluating the 
performance of the forecast layer.

In general it could be seen that:
* The forecasted inundation extent generally over-predicted when compared to GFM
* Because forecast data is coarser resolution
    * Smoothed representation of floodplain topography, e.g. doesn't represent levees
* Therefore you need to be careful regarding the precision for which the forecast information can be used
    * It can highlight general floodplain areas at risk but be careful when identifying specific towns/cities affected